# SI Tables S1 and S2: the Pareto plot data

Each method's total compute time and CDCl3 test RMSE, for proton (S1) and carbon (S2): a curated
55-row subset (MagNET, one AIMNet2-geometry row, and the DFT grid on PBE0/cc-pVTZ geometries).

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import numpy as np
import pandas as pd

import delta22
import paths

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def document_path(name):
    os.makedirs("documents", exist_ok=True)
    return os.path.join("documents", name)

In [ ]:
# Load the flat DFT and MagNET tables and the timing tables.
dft = delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False)
nn = delta22.load_query_df_nn(DELTA22_HDF5, XLSX, verbose=False)
dft_gas_timings, nn_timings = delta22.load_pareto_timings(DELTA22_HDF5)

# One point per method/basis/geometry/nucleus/solvent, plus the solvent-averaged rows, each with its
# mean test RMSE and total compute time. MagNET appears as a single method (aimnet2, basis "N/A").
# Figure 2A uses 100 seeded splits, not the 250 the other delta-22 panels use, and trains on the
# first 10 shuffled solutes / tests on the rest; fig2a_pareto_points handles that split convention.
PARETO_N_SPLITS = 100
points = delta22.fig2a_pareto_points(dft, nn, dft_gas_timings, nn_timings, n_splits=PARETO_N_SPLITS)
print(points["nmr_method"].nunique(), "methods;",
      "MagNET total time", float(points.query("nmr_method=='MagNET'")["total_time"].iloc[0]), "s")

## Tables S1 and S2

In [ ]:
# full column set matching the SI tables: identity + fitting RMSE + the three time components and
# their log10 (all solvents here are chloroform). geometry_time and nmr_time sum to total_time.
COLUMNS = ["geometry_type", "nmr_method", "basis", "solvent", "fitting_RMSE",
           "geometry_time", "nmr_time", "total_time", "log10_total_time"]

curated = {}
for nucleus, label in [("H", "S1"), ("C", "S2")]:
    sub = delta22.pareto_table_curated(points, nucleus).copy()
    sub["log10_total_time"] = np.log10(sub["total_time"])
    sub = sub[COLUMNS]
    curated[label] = sub
    print(f"Table {label} ({nucleus}): {len(sub)} rows x {sub.shape[1]} columns")
    print(sub.to_string(index=False), "\n")

# write the two tables to this notebook's documents/ folder, one sheet per SI table
out = document_path("si_table_s01_s02_pareto.xlsx")
with pd.ExcelWriter(out) as writer:
    curated["S1"].to_excel(writer, sheet_name="Table S1 (1H)", index=False)
    curated["S2"].to_excel(writer, sheet_name="Table S2 (13C)", index=False)
print("wrote", os.path.relpath(out, REPO))